# Retail Sales Analysis — Portfolio Notebook v1.0

**ArtoWare Indonesia**

A business-facing analytical narrative for the Retail Sales Analysis portfolio project.

**Version:** v1.0.0 portfolio notebook  
**Dataset:** Superstore retail sales dataset  
**Scope:** data quality → business metrics → business analysis → visual evidence → business implications

> Run from the repository root with the project dependencies installed.

## 1. Business Problem & Objectives

Retail sales data can show strong revenue performance while hiding differences in profitability across categories, regions, customers, and products.

This analysis asks:

1. What is the overall sales and profitability performance?
2. Which categories and regions contribute most to sales and profit?
3. How does performance change over time?
4. Which customers and products have the greatest commercial impact?
5. Where are the main profitability risks or concentration points?

## 2. Load Data & Validate the Analysis Boundary

The repository establishes a canonical retail schema before analysis. Downstream analysis therefore uses the same data boundary as the main application.

In [ ]:
from pathlib import Path
from IPython.display import display, Image
import pandas as pd
from config import DATASET_FILE
from src.loader import load_dataset
from src.schema import normalize_dataset
from src.cleaning import DataCleaner
from src.business_metrics import BusinessMetrics
from src.insights import BusinessInsights
from src.visualization import Visualizer

CLEAN_DATA = Path('data/processed/superstore_clean.csv')
IMAGE_DIR = Path('images')
print('Raw dataset:', DATASET_FILE)
print('Codebase: v0.6.0 | Portfolio notebook: v1.0.0')

In [ ]:
df = normalize_dataset(load_dataset(DATASET_FILE))
print('Rows: {:,}'.format(len(df)))
print('Columns: {:,}'.format(len(df.columns)))
display(df.head())

In [ ]:
missing = df.isna().sum()
print('Duplicate rows: {:,}'.format(df.duplicated().sum()))
display(missing[missing > 0].to_frame('Missing Values') if (missing > 0).any() else pd.DataFrame({'Status':['No missing values detected']}))

## 3. Data Preparation

Cleaning is performed through the reusable DataCleaner rather than maintaining a separate notebook implementation.

In [ ]:
cleaned_df = DataCleaner(df, CLEAN_DATA).run()
print('Cleaned rows: {:,}'.format(len(cleaned_df)))
print('Clean dataset:', CLEAN_DATA)

## 4. Business Performance

The KPI layer establishes the overall commercial baseline.

In [ ]:
kpis = BusinessMetrics(cleaned_df).run()
kpi_table = pd.DataFrame({
    'Metric':['Total Sales','Total Profit','Profit Margin','Total Orders','Total Customers','Total Products'],
    'Value':[
        '${:,.2f}'.format(kpis['total_sales']),
        '${:,.2f}'.format(kpis['total_profit']),
        '{:.2f}%'.format(kpis['profit_margin']),
        '{:,}'.format(kpis['total_orders']),
        '{:,}'.format(kpis['total_customers']),
        '{:,}'.format(kpis['total_products'])]})
display(kpi_table)

## 5. Business Analysis

BusinessInsights evaluates category, region, monthly, customer, product, contribution, and profitability dimensions.

In [ ]:
results = BusinessInsights(cleaned_df).run()
category = results['category']['summary'].sort_values('Sales', ascending=False)
region = results['region']['summary'].sort_values('Sales', ascending=False)
print('Category performance')
display(category)
print('Regional performance')
display(region)
print('Top 10 customers by sales')
display(results['customer']['top_10_customers'].to_frame('Sales'))
print('Top 10 products by sales')
display(results['product']['products'].sort_values('Sales', ascending=False).head(10))

In [ ]:
print('Category contribution')
display(results['contribution']['category'])
print('Regional contribution')
display(results['contribution']['region'])
profitability = results['profitability']
print('Lowest-margin category:', profitability['lowest_margin_category'])
print('Lowest-margin region:', profitability['lowest_margin_region'])
print('Loss-making products: {:,}'.format(profitability['loss_making_product_count']))

### Key Analytical Findings

- **Scale and profitability are different dimensions.** Sales leadership should be read together with profit contribution and margin.
- **Regional performance varies.** Sales, profit, and margin can tell different stories.
- **Concentration matters.** Top customers and products show where commercial performance is concentrated.
- **Time matters.** Monthly trends identify stronger and weaker periods for further investigation.
- **Profitability risks require targeted analysis.** Low-margin dimensions and loss-making products can be hidden inside strong aggregate sales.

These are descriptive observations from the dataset, not causal claims.

## 6. Visual Evidence

The notebook uses the repository's reusable Visualizer and surfaces a focused subset of the static artifacts.

In [ ]:
visualizer = Visualizer(cleaned_df, output_dir=IMAGE_DIR)
visualizer.sales_by_category(cleaned_df)
visualizer.sales_by_region(cleaned_df)
visualizer.monthly_sales_trend(cleaned_df)
visualizer.profit_analysis(cleaned_df)
visualizer.profit_margin_by_category(cleaned_df)

figures = [
    ('Sales by Category', IMAGE_DIR/'sales_by_category.png'),
    ('Sales by Region', IMAGE_DIR/'sales_by_region.png'),
    ('Monthly Sales Trend', IMAGE_DIR/'monthly_sales_trend.png'),
    ('Profit by Category', IMAGE_DIR/'profit_by_category.png'),
    ('Profit Margin by Category', IMAGE_DIR/'profit_margin_by_category.png')]
for title, path in figures:
    print(title)
    display(Image(filename=str(path)))

## 7. Business Insights & Implications

The combined evidence shows why retail performance should not be evaluated through sales alone. Profit, margin, contribution, concentration, and time trends provide additional context for business investigation.

The analysis identifies areas to investigate further, including low-margin dimensions, loss-making products, and concentration among leading customers or products. It does not claim causal explanations from the dataset alone.

## 8. Conclusion

The final project combines reusable data cleaning and validation, business KPIs, structured insights, static visualization, an interactive Plotly dashboard, automated regression tests, and controlled dataset schema mapping.

The notebook is the **portfolio storytelling layer**. The src/ package remains the reusable engineering layer.

### Further exploration

Interactive dashboard: `output/interactive/interactive_dashboard.html`

The README documents the complete pipeline and test workflow.